# Task 2 ? Final Official-Test Evaluation and Model Comparison

This notebook performs the single common final evaluation of
the three already-trained Task 2 sentiment classifiers:

1. `baseline_mean_pool`
2. `textcnn`
3. `bigru_attention`

The evaluation protocol was frozen and pushed before official
test access:

`configs/final_evaluation.yaml`

Frozen protocol SHA-256:

`ab26c7dc9382589c1e5358a94d0a0ce0c1573a208ec29b9f81d541e415b61d91`

Scientific constraints:

- no training occurs here;
- no model selection occurs here;
- no threshold selection occurs here;
- decision threshold remains 0.5;
- all models use the identical 38,000 official test rows;
- model architectures are copied exactly from the verified
  executed training notebooks;
- test forward passes preserve the training-time autocast /
  sigmoid convention;
- 2,000 bootstrap replicates each sample 38,000 test rows
  with replacement;
- paired McNemar comparisons use the same test rows;
- Holm correction covers the two planned comparisons;
- robustness-slice thresholds were derived from training only
  and frozen before test access.

The notebook is source-audited before its first execution.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from IPython.display import display
from scipy.stats import chi2
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch import nn
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
)


def find_repo_root():
    current = Path.cwd().resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            candidate
            / "task2_sentiment"
            / "Anshika_Goel"
        ).is_dir():
            return candidate

    raise RuntimeError(
        "Unable to locate repository root."
    )


def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:
        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


ROOT = find_repo_root()

MEMBER = (
    ROOT
    / "task2_sentiment"
    / "Anshika_Goel"
)

PROCESSED = (
    ROOT
    / "task2_sentiment"
    / "data"
    / "processed"
    / "Anshika_Goel"
)

CONFIG_PATH = (
    MEMBER
    / "configs"
    / "sentiment.yaml"
)

PROTOCOL_PATH = (
    MEMBER
    / "configs"
    / "final_evaluation.yaml"
)

VOCABULARY_PATH = (
    MEMBER
    / "outputs"
    / "metrics"
    / "vocabulary.json"
)

SPLIT_MANIFEST_PATH = (
    MEMBER
    / "outputs"
    / "metrics"
    / "split_manifest.json"
)

METRICS_DIRECTORY = (
    MEMBER
    / "outputs"
    / "metrics"
)

PLOTS_DIRECTORY = (
    MEMBER
    / "outputs"
    / "plots"
)

PREDICTIONS_DIRECTORY = (
    MEMBER
    / "outputs"
    / "predictions"
)

CHECKPOINT_DIRECTORY = (
    MEMBER
    / "checkpoints"
)


EXPECTED_PROTOCOL_SHA256 = (
    "ab26c7dc9382589c1e5358a94d0a0ce0"
    "c1573a208ec29b9f81d541e415b61d91"
)

EXPECTED_CONFIG_SHA256 = (
    "07f59eeb49af2fbdcce4f301e76625ac"
    "2c4ae8150eb1b89ef55862b6b5c9758b"
)

EXPECTED_VOCABULARY_SHA256 = (
    "0d06c4dc3a6df1ae90b86ec6726a0caf"
    "b81fe9eab32da75fa74b395258729124"
)

EXPECTED_SPLIT_MANIFEST_SHA256 = (
    "c782f45fb20eec7e3d12b1197baa346d"
    "aeb5cb4b6adabf81d4cf5bf09584c28c"
)


EXPECTED_CHECKPOINT_SHA256 = {
    "baseline_mean_pool": (
        "4ba3b1585ff34e40cf26d74dd7a56f59"
        "d38fc8010e795046772472bf72bae724"
    ),
    "textcnn": (
        "6f65361355ab9219177dec3b6cb482d7"
        "f7e3103ecf0f9110c3eefda42b7e2c1a"
    ),
    "bigru_attention": (
        "da6841121a69416601fbc9fde5e2e9c3"
        "2fb4cc5da4cd4a9efb758e35ab66657a"
    ),
}


EXPECTED_PARAMETER_COUNTS = {
    "baseline_mean_pool": 6_400_129,
    "textcnn": 6_597_377,
    "bigru_attention": 6_631_425,
}


print(
    "Repository root:",
    ROOT,
)

print(
    "Python platform:",
    platform.platform(),
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "CUDA available:",
    torch.cuda.is_available(),
)


Repository root: C:\Users\aradh\Desktop\DATA 266 Generative AI and LLM\Lab1
Python platform: Windows-10-10.0.26200-SP0
PyTorch: 2.13.0+cu126
CUDA available: True


In [2]:
training_configuration = yaml.safe_load(
    CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

evaluation_protocol = yaml.safe_load(
    PROTOCOL_PATH.read_text(
        encoding="utf-8"
    )
)

vocabulary_payload = json.loads(
    VOCABULARY_PATH.read_text(
        encoding="utf-8"
    )
)

split_manifest = json.loads(
    SPLIT_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    sha256_file(
        PROTOCOL_PATH
    )
    == EXPECTED_PROTOCOL_SHA256
)

assert (
    sha256_file(
        CONFIG_PATH
    )
    == EXPECTED_CONFIG_SHA256
)

assert (
    sha256_file(
        VOCABULARY_PATH
    )
    == EXPECTED_VOCABULARY_SHA256
)

assert (
    sha256_file(
        SPLIT_MANIFEST_PATH
    )
    == EXPECTED_SPLIT_MANIFEST_SHA256
)


assert (
    evaluation_protocol[
        "protocol"
    ][
        "status"
    ]
    == "frozen_before_official_test_access"
)

assert (
    evaluation_protocol[
        "official_test_policy"
    ][
        "expected_examples"
    ]
    == 38_000
)

assert (
    evaluation_protocol[
        "official_test_policy"
    ][
        "used_for_threshold_selection"
    ]
    is False
)

assert (
    evaluation_protocol[
        "official_test_policy"
    ][
        "used_for_slice_threshold_selection"
    ]
    is False
)

assert (
    evaluation_protocol[
        "classification"
    ][
        "decision_threshold"
    ]
    == 0.5
)


MODEL_ORDER = list(
    evaluation_protocol[
        "models"
    ]
)


assert MODEL_ORDER == [
    "baseline_mean_pool",
    "textcnn",
    "bigru_attention",
]


CHECKPOINT_PATHS = {
    "baseline_mean_pool": (
        CHECKPOINT_DIRECTORY
        / "baseline_mean_pool_best.pt"
    ),
    "textcnn": (
        CHECKPOINT_DIRECTORY
        / "textcnn_best.pt"
    ),
    "bigru_attention": (
        CHECKPOINT_DIRECTORY
        / "bigru_attention_best.pt"
    ),
}


TRAINING_METRICS_PATHS = {
    model_name: (
        METRICS_DIRECTORY
        / f"{model_name}_metrics.json"
    )
    for model_name
    in MODEL_ORDER
}

TRAINING_HISTORY_PATHS = {
    model_name: (
        METRICS_DIRECTORY
        / f"{model_name}_history.json"
    )
    for model_name
    in MODEL_ORDER
}


training_metrics_artifacts = {
    model_name: json.loads(
        TRAINING_METRICS_PATHS[
            model_name
        ].read_text(
            encoding="utf-8"
        )
    )
    for model_name
    in MODEL_ORDER
}


training_history_artifacts = {
    model_name: json.loads(
        TRAINING_HISTORY_PATHS[
            model_name
        ].read_text(
            encoding="utf-8"
        )
    )
    for model_name
    in MODEL_ORDER
}


for model_name in MODEL_ORDER:
    assert (
        sha256_file(
            CHECKPOINT_PATHS[
                model_name
            ]
        )
        == EXPECTED_CHECKPOINT_SHA256[
            model_name
        ]
    )

    assert (
        training_metrics_artifacts[
            model_name
        ][
            "official_test_used"
        ]
        is False
    )


print(
    "FROZEN PRE-TEST ARTIFACT CONTRACT VERIFIED"
)


FROZEN PRE-TEST ARTIFACT CONTRACT VERIFIED


In [3]:
class MeanPoolSentimentClassifier(
    nn.Module
):
    def __init__(
        self,
        vocabulary_size,
        embedding_dimension,
        padding_index,
        dropout_probability,
    ):
        super().__init__()

        self.padding_index = int(
            padding_index
        )

        self.embedding = nn.Embedding(
            num_embeddings=(
                vocabulary_size
            ),
            embedding_dim=(
                embedding_dimension
            ),
            padding_idx=(
                self.padding_index
            ),
        )

        self.dropout = nn.Dropout(
            p=dropout_probability
        )

        self.classifier = nn.Linear(
            embedding_dimension,
            1,
        )

        self.reset_parameters()


    def reset_parameters(self):
        nn.init.normal_(
            self.embedding.weight,
            mean=0.0,
            std=0.02,
        )

        with torch.no_grad():
            self.embedding.weight[
                self.padding_index
            ].zero_()

        nn.init.xavier_uniform_(
            self.classifier.weight
        )

        nn.init.zeros_(
            self.classifier.bias
        )


    def forward(self, input_ids):
        mask = (
            input_ids
            != self.padding_index
        ).unsqueeze(-1)

        embedded = self.embedding(
            input_ids
        )

        masked_embeddings = (
            embedded
            * mask.to(
                dtype=embedded.dtype
            )
        )

        summed = (
            masked_embeddings.sum(
                dim=1
            )
        )

        token_counts = (
            mask.sum(
                dim=1
            )
            .clamp(min=1)
            .to(
                dtype=embedded.dtype
            )
        )

        pooled = (
            summed
            / token_counts
        )

        pooled = self.dropout(
            pooled
        )

        logits = (
            self.classifier(
                pooled
            )
            .squeeze(-1)
        )

        return logits


class TextCNNSentimentClassifier(
    nn.Module
):
    def __init__(
        self,
        vocabulary_size,
        embedding_dimension,
        padding_index,
        kernel_sizes,
        filters_per_kernel,
        dropout_probability,
    ):
        super().__init__()

        self.padding_index = int(
            padding_index
        )

        self.kernel_sizes = tuple(
            int(kernel_size)
            for kernel_size in kernel_sizes
        )

        self.filters_per_kernel = int(
            filters_per_kernel
        )

        self.embedding = nn.Embedding(
            num_embeddings=(
                vocabulary_size
            ),
            embedding_dim=(
                embedding_dimension
            ),
            padding_idx=(
                self.padding_index
            ),
        )

        self.convolutions = nn.ModuleList(
            [
                nn.Conv1d(
                    in_channels=(
                        embedding_dimension
                    ),
                    out_channels=(
                        self.filters_per_kernel
                    ),
                    kernel_size=(
                        kernel_size
                    ),
                )
                for kernel_size
                in self.kernel_sizes
            ]
        )

        concatenated_dimension = (
            self.filters_per_kernel
            * len(
                self.kernel_sizes
            )
        )

        self.dropout = nn.Dropout(
            p=dropout_probability
        )

        self.classifier = nn.Linear(
            concatenated_dimension,
            1,
        )

        self.reset_parameters()


    def reset_parameters(self):
        nn.init.normal_(
            self.embedding.weight,
            mean=0.0,
            std=0.02,
        )

        with torch.no_grad():
            self.embedding.weight[
                self.padding_index
            ].zero_()

        for convolution in (
            self.convolutions
        ):
            nn.init.kaiming_uniform_(
                convolution.weight,
                nonlinearity="relu",
            )

            if (
                convolution.bias
                is not None
            ):
                nn.init.zeros_(
                    convolution.bias
                )

        nn.init.xavier_uniform_(
            self.classifier.weight
        )

        nn.init.zeros_(
            self.classifier.bias
        )


    def forward(
        self,
        input_ids,
    ):
        token_mask = (
            input_ids
            != self.padding_index
        )

        embedded = self.embedding(
            input_ids
        )

        convolution_input = (
            embedded.transpose(
                1,
                2,
            )
        )

        mask_input = (
            token_mask
            .unsqueeze(1)
            .to(
                dtype=(
                    convolution_input.dtype
                )
            )
        )

        pooled_features = []

        for (
            convolution,
            kernel_size,
        ) in zip(
            self.convolutions,
            self.kernel_sizes,
        ):
            activations = torch.relu(
                convolution(
                    convolution_input
                )
            )

            valid_windows = (
                torch.nn.functional.max_pool1d(
                    mask_input,
                    kernel_size=(
                        kernel_size
                    ),
                    stride=1,
                )
                > 0
            )

            minimum_value = torch.finfo(
                activations.dtype
            ).min

            masked_activations = (
                activations.masked_fill(
                    ~valid_windows,
                    minimum_value,
                )
            )

            pooled = (
                masked_activations.amax(
                    dim=2
                )
            )

            has_valid_window = (
                valid_windows.any(
                    dim=2
                )
            )

            pooled = torch.where(
                has_valid_window,
                pooled,
                torch.zeros_like(
                    pooled
                ),
            )

            pooled_features.append(
                pooled
            )

        concatenated = torch.cat(
            pooled_features,
            dim=1,
        )

        concatenated = self.dropout(
            concatenated
        )

        logits = (
            self.classifier(
                concatenated
            )
            .squeeze(-1)
        )

        return logits


class BiGRUAttentionSentimentClassifier(
    nn.Module
):
    def __init__(
        self,
        vocabulary_size,
        embedding_dimension,
        padding_index,
        hidden_dimension,
        recurrent_layers,
        bidirectional,
        attention_dimension,
        dropout_probability,
    ):
        super().__init__()

        self.padding_index = int(
            padding_index
        )

        self.hidden_dimension = int(
            hidden_dimension
        )

        self.recurrent_layers = int(
            recurrent_layers
        )

        self.bidirectional = bool(
            bidirectional
        )

        self.attention_dimension = int(
            attention_dimension
        )

        self.num_directions = (
            2
            if self.bidirectional
            else 1
        )

        self.recurrent_output_dimension = (
            self.hidden_dimension
            * self.num_directions
        )

        self.embedding = nn.Embedding(
            num_embeddings=(
                vocabulary_size
            ),
            embedding_dim=(
                embedding_dimension
            ),
            padding_idx=(
                self.padding_index
            ),
        )

        self.gru = nn.GRU(
            input_size=(
                embedding_dimension
            ),
            hidden_size=(
                self.hidden_dimension
            ),
            num_layers=(
                self.recurrent_layers
            ),
            batch_first=True,
            bidirectional=(
                self.bidirectional
            ),
            dropout=(
                dropout_probability
                if self.recurrent_layers > 1
                else 0.0
            ),
        )

        self.attention_projection = nn.Linear(
            self.recurrent_output_dimension,
            self.attention_dimension,
            bias=True,
        )

        self.attention_score = nn.Linear(
            self.attention_dimension,
            1,
            bias=False,
        )

        self.dropout = nn.Dropout(
            p=dropout_probability
        )

        self.classifier = nn.Linear(
            self.recurrent_output_dimension,
            1,
        )

        self.reset_parameters()


    def reset_parameters(self):
        nn.init.normal_(
            self.embedding.weight,
            mean=0.0,
            std=0.02,
        )

        with torch.no_grad():
            self.embedding.weight[
                self.padding_index
            ].zero_()

        for name, parameter in (
            self.gru.named_parameters()
        ):
            if (
                "weight_ih"
                in name
            ):
                for gate_weight in (
                    parameter.chunk(
                        3,
                        dim=0,
                    )
                ):
                    nn.init.xavier_uniform_(
                        gate_weight
                    )

            elif (
                "weight_hh"
                in name
            ):
                for gate_weight in (
                    parameter.chunk(
                        3,
                        dim=0,
                    )
                ):
                    nn.init.orthogonal_(
                        gate_weight
                    )

            elif (
                "bias"
                in name
            ):
                nn.init.zeros_(
                    parameter
                )

        nn.init.xavier_uniform_(
            self.attention_projection.weight
        )

        nn.init.zeros_(
            self.attention_projection.bias
        )

        nn.init.xavier_uniform_(
            self.attention_score.weight
        )

        nn.init.xavier_uniform_(
            self.classifier.weight
        )

        nn.init.zeros_(
            self.classifier.bias
        )


    def forward(
        self,
        input_ids,
    ):
        token_mask = (
            input_ids
            != self.padding_index
        )

        sequence_lengths = (
            token_mask.sum(
                dim=1
            )
            .to(
                dtype=torch.int64
            )
        )

        packed_lengths = (
            sequence_lengths
            .clamp_min(
                1
            )
            .cpu()
        )

        embedded = self.embedding(
            input_ids
        )

        packed_input = (
            nn.utils.rnn.pack_padded_sequence(
                embedded,
                lengths=packed_lengths,
                batch_first=True,
                enforce_sorted=False,
            )
        )

        packed_output, _ = self.gru(
            packed_input
        )

        recurrent_output, _ = (
            nn.utils.rnn.pad_packed_sequence(
                packed_output,
                batch_first=True,
                total_length=(
                    input_ids.shape[
                        1
                    ]
                ),
            )
        )

        attention_hidden = torch.tanh(
            self.attention_projection(
                recurrent_output
            )
        )

        attention_scores = (
            self.attention_score(
                attention_hidden
            )
            .squeeze(-1)
        )

        minimum_value = torch.finfo(
            attention_scores.dtype
        ).min

        masked_attention_scores = (
            attention_scores.masked_fill(
                ~token_mask,
                minimum_value,
            )
        )

        attention_weights = torch.softmax(
            masked_attention_scores,
            dim=1,
        )

        attention_weights = (
            attention_weights
            * token_mask.to(
                dtype=(
                    attention_weights.dtype
                )
            )
        )

        attention_normalizer = (
            attention_weights.sum(
                dim=1,
                keepdim=True,
            )
        )

        safe_normalizer = torch.where(
            attention_normalizer > 0,
            attention_normalizer,
            torch.ones_like(
                attention_normalizer
            ),
        )

        attention_weights = (
            attention_weights
            / safe_normalizer
        )

        context = torch.bmm(
            attention_weights.unsqueeze(
                1
            ),
            recurrent_output,
        ).squeeze(
            1
        )

        context = self.dropout(
            context
        )

        logits = (
            self.classifier(
                context
            )
            .squeeze(-1)
        )

        return logits


In [4]:
def expected_calibration_error(
    y_true,
    probabilities,
    bins=15,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        bins + 1,
    )

    predicted_labels = (
        probabilities >= 0.5
    ).astype(np.int64)

    confidences = np.maximum(
        probabilities,
        1.0 - probabilities,
    )

    correctness = (
        predicted_labels == y_true
    ).astype(np.float64)

    ece = 0.0

    for bin_index in range(bins):
        lower = bin_edges[
            bin_index
        ]

        upper = bin_edges[
            bin_index + 1
        ]

        if bin_index == bins - 1:
            mask = (
                (confidences >= lower)
                & (confidences <= upper)
            )
        else:
            mask = (
                (confidences >= lower)
                & (confidences < upper)
            )

        if not np.any(mask):
            continue

        bin_fraction = float(
            np.mean(mask)
        )

        bin_accuracy = float(
            np.mean(
                correctness[mask]
            )
        )

        bin_confidence = float(
            np.mean(
                confidences[mask]
            )
        )

        ece += (
            bin_fraction
            * abs(
                bin_accuracy
                - bin_confidence
            )
        )

    return float(ece)


def classification_metrics(
    labels,
    probabilities,
    threshold=0.5,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    (
        macro_precision,
        macro_recall,
        macro_f1,
        _,
    ) = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    (
        micro_precision,
        micro_recall,
        micro_f1,
        _,
    ) = precision_recall_fscore_support(
        labels,
        predictions,
        average="micro",
        zero_division=0,
    )

    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _,
    ) = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    confusion = confusion_matrix(
        labels,
        predictions,
    )

    return {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "macro_precision": float(
            macro_precision
        ),
        "micro_precision": float(
            micro_precision
        ),
        "weighted_precision": float(
            weighted_precision
        ),
        "macro_recall": float(
            macro_recall
        ),
        "micro_recall": float(
            micro_recall
        ),
        "weighted_recall": float(
            weighted_recall
        ),
        "macro_f1": float(
            macro_f1
        ),
        "micro_f1": float(
            micro_f1
        ),
        "weighted_f1": float(
            weighted_f1
        ),
        "mcc": float(
            matthews_corrcoef(
                labels,
                predictions,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                labels,
                probabilities,
            )
        ),
        "ece_15_bins": float(
            expected_calibration_error(
                labels,
                probabilities,
                bins=15,
            )
        ),
        "confusion_matrix": (
            confusion.tolist()
        ),
        "threshold": float(
            threshold
        ),
    }


In [5]:
def metric_value(
    metrics,
    *candidate_keys,
):
    for key in candidate_keys:
        if key in metrics:
            return metrics[
                key
            ]

    raise KeyError(
        "None of the metric aliases exist: "
        + repr(
            candidate_keys
        )
    )


def canonical_metrics(
    raw_metrics,
):
    return {
        "accuracy": float(
            metric_value(
                raw_metrics,
                "accuracy",
            )
        ),
        "precision_macro": float(
            metric_value(
                raw_metrics,
                "precision_macro",
                "macro_precision",
            )
        ),
        "precision_micro": float(
            metric_value(
                raw_metrics,
                "precision_micro",
                "micro_precision",
            )
        ),
        "precision_weighted": float(
            metric_value(
                raw_metrics,
                "precision_weighted",
                "weighted_precision",
            )
        ),
        "recall_macro": float(
            metric_value(
                raw_metrics,
                "recall_macro",
                "macro_recall",
            )
        ),
        "recall_micro": float(
            metric_value(
                raw_metrics,
                "recall_micro",
                "micro_recall",
            )
        ),
        "recall_weighted": float(
            metric_value(
                raw_metrics,
                "recall_weighted",
                "weighted_recall",
            )
        ),
        "f1_macro": float(
            metric_value(
                raw_metrics,
                "f1_macro",
                "macro_f1",
            )
        ),
        "f1_micro": float(
            metric_value(
                raw_metrics,
                "f1_micro",
                "micro_f1",
            )
        ),
        "f1_weighted": float(
            metric_value(
                raw_metrics,
                "f1_weighted",
                "weighted_f1",
            )
        ),
        "mcc": float(
            metric_value(
                raw_metrics,
                "mcc",
            )
        ),
        "roc_auc": float(
            metric_value(
                raw_metrics,
                "roc_auc",
            )
        ),
        "pr_auc": float(
            metric_value(
                raw_metrics,
                "pr_auc",
            )
        ),
        "brier_score": float(
            metric_value(
                raw_metrics,
                "brier_score",
                "brier",
            )
        ),
        "ece_15_bins": float(
            metric_value(
                raw_metrics,
                "ece_15_bins",
                "ece",
            )
        ),
        "confusion_matrix": (
            metric_value(
                raw_metrics,
                "confusion_matrix",
            )
        ),
    }


def recursive_values_for_key(
    payload,
    target_key,
):
    values = []

    if isinstance(
        payload,
        dict,
    ):
        for key, value in (
            payload.items()
        ):
            if key == target_key:
                values.append(
                    value
                )

            values.extend(
                recursive_values_for_key(
                    value,
                    target_key,
                )
            )

    elif isinstance(
        payload,
        list,
    ):
        for value in payload:
            values.extend(
                recursive_values_for_key(
                    value,
                    target_key,
                )
            )

    return values


def first_numeric_by_keys(
    payload,
    candidate_keys,
):
    for candidate_key in candidate_keys:
        values = (
            recursive_values_for_key(
                payload,
                candidate_key,
            )
        )

        numeric = [
            value
            for value in values
            if (
                isinstance(
                    value,
                    (int, float),
                )
                and not isinstance(
                    value,
                    bool,
                )
            )
        ]

        if numeric:
            return float(
                numeric[0]
            )

    return None


def history_rows(
    payload,
):
    if isinstance(
        payload,
        list,
    ):
        return payload

    if isinstance(
        payload,
        dict,
    ):
        for key in [
            "history",
            "epochs",
        ]:
            rows = payload.get(
                key
            )

            if isinstance(
                rows,
                list,
            ):
                return rows

    return []


training_resource_summary = {}


for model_name in MODEL_ORDER:
    artifact = (
        training_metrics_artifacts[
            model_name
        ]
    )

    history = history_rows(
        training_history_artifacts[
            model_name
        ]
    )

    training_seconds = (
        first_numeric_by_keys(
            artifact,
            [
                "total_training_wall_seconds",
                "total_training_seconds",
                "overall_training_seconds",
                "training_seconds",
                "wall_clock_seconds",
                "wall_clock_training_seconds",
            ],
        )
    )

    mean_examples_per_second = (
        first_numeric_by_keys(
            artifact,
            [
                "mean_epoch_train_examples_per_second",
                "mean_train_examples_per_second",
                "mean_examples_per_second",
            ],
        )
    )

    if (
        mean_examples_per_second
        is None
    ):
        epoch_values = []

        for row in history:
            if not isinstance(
                row,
                dict,
            ):
                continue

            for key in [
                "train_examples_per_second",
                "examples_per_second",
            ]:
                if key in row:
                    epoch_values.append(
                        float(
                            row[
                                key
                            ]
                        )
                    )

                    break

        if epoch_values:
            mean_examples_per_second = (
                float(
                    np.mean(
                        epoch_values
                    )
                )
            )

    peak_gpu_allocated_mb = (
        first_numeric_by_keys(
            artifact,
            [
                "peak_gpu_allocated_mb",
                "peak_allocated_mb",
            ],
        )
    )

    peak_gpu_reserved_mb = (
        first_numeric_by_keys(
            artifact,
            [
                "peak_gpu_reserved_mb",
                "peak_reserved_mb",
            ],
        )
    )


    training_resource_summary[
        model_name
    ] = {
        "parameter_count": int(
            EXPECTED_PARAMETER_COUNTS[
                model_name
            ]
        ),
        "training_seconds": (
            training_seconds
        ),
        "mean_train_examples_per_second": (
            mean_examples_per_second
        ),
        "peak_gpu_allocated_mb": (
            peak_gpu_allocated_mb
        ),
        "peak_gpu_reserved_mb": (
            peak_gpu_reserved_mb
        ),
    }


print(
    json.dumps(
        training_resource_summary,
        indent=2,
    )
)


{
  "baseline_mean_pool": {
    "parameter_count": 6400129,
    "training_seconds": 72.6784261999419,
    "mean_train_examples_per_second": 36636.53049082684,
    "peak_gpu_allocated_mb": 269.08642578125,
    "peak_gpu_reserved_mb": 308.0
  },
  "textcnn": {
    "parameter_count": 6597377,
    "training_seconds": 358.8450522000203,
    "mean_train_examples_per_second": 7310.545900300038,
    "peak_gpu_allocated_mb": 308.88232421875,
    "peak_gpu_reserved_mb": 402.0
  },
  "bigru_attention": {
    "parameter_count": 6631425,
    "training_seconds": 1545.914335199981,
    "mean_train_examples_per_second": 1401.1242295852849,
    "peak_gpu_allocated_mb": 223.447265625,
    "peak_gpu_reserved_mb": 486.0
  }
}


In [6]:
def metrics_from_binary_counts(
    tn,
    fp,
    fn,
    tp,
):
    total = (
        tn
        + fp
        + fn
        + tp
    )

    accuracy = (
        (tn + tp)
        / total
    )

    positive_denominator = (
        2 * tp
        + fp
        + fn
    )

    negative_denominator = (
        2 * tn
        + fp
        + fn
    )

    positive_f1 = (
        0.0
        if positive_denominator == 0
        else (
            2 * tp
            / positive_denominator
        )
    )

    negative_f1 = (
        0.0
        if negative_denominator == 0
        else (
            2 * tn
            / negative_denominator
        )
    )

    macro_f1 = (
        positive_f1
        + negative_f1
    ) / 2.0

    denominator = math.sqrt(
        (tp + fp)
        * (tp + fn)
        * (tn + fp)
        * (tn + fn)
    )

    mcc = (
        0.0
        if denominator == 0
        else (
            (
                tp * tn
                - fp * fn
            )
            / denominator
        )
    )

    return (
        float(
            accuracy
        ),
        float(
            macro_f1
        ),
        float(
            mcc
        ),
    )


def bootstrap_confidence_intervals(
    labels,
    predictions_by_model,
    replicates,
    confidence_level,
    seed,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    model_names = list(
        predictions_by_model.keys()
    )

    prediction_matrix = np.stack(
        [
            np.asarray(
                predictions_by_model[
                    model_name
                ],
                dtype=np.int64,
            )
            for model_name
            in model_names
        ],
        axis=0,
    )

    sample_count = int(
        labels.shape[0]
    )

    generator = (
        np.random.default_rng(
            seed
        )
    )

    values = {
        model_name: {
            "accuracy": np.empty(
                replicates,
                dtype=np.float64,
            ),
            "macro_f1": np.empty(
                replicates,
                dtype=np.float64,
            ),
            "mcc": np.empty(
                replicates,
                dtype=np.float64,
            ),
        }
        for model_name
        in model_names
    }


    for replicate_index in range(
        replicates
    ):
        indices = generator.integers(
            low=0,
            high=sample_count,
            size=sample_count,
            dtype=np.int64,
        )

        sampled_labels = labels[
            indices
        ]

        for (
            model_index,
            model_name,
        ) in enumerate(
            model_names
        ):
            sampled_predictions = (
                prediction_matrix[
                    model_index,
                    indices,
                ]
            )

            encoded = (
                sampled_labels * 2
                + sampled_predictions
            )

            counts = np.bincount(
                encoded,
                minlength=4,
            )

            (
                accuracy,
                macro_f1,
                mcc,
            ) = (
                metrics_from_binary_counts(
                    tn=int(
                        counts[0]
                    ),
                    fp=int(
                        counts[1]
                    ),
                    fn=int(
                        counts[2]
                    ),
                    tp=int(
                        counts[3]
                    ),
                )
            )

            values[
                model_name
            ][
                "accuracy"
            ][
                replicate_index
            ] = accuracy

            values[
                model_name
            ][
                "macro_f1"
            ][
                replicate_index
            ] = macro_f1

            values[
                model_name
            ][
                "mcc"
            ][
                replicate_index
            ] = mcc


    alpha = (
        1.0
        - confidence_level
    )

    lower_q = (
        alpha / 2.0
    )

    upper_q = (
        1.0
        - alpha / 2.0
    )

    result = {}


    for model_name in model_names:
        result[
            model_name
        ] = {}

        for (
            metric_name,
            samples,
        ) in values[
            model_name
        ].items():
            lower, upper = np.quantile(
                samples,
                [
                    lower_q,
                    upper_q,
                ],
                method="linear",
            )

            result[
                model_name
            ][
                metric_name
            ] = {
                "lower": float(
                    lower
                ),
                "upper": float(
                    upper
                ),
            }


    return result


def mcnemar_continuity_corrected(
    labels,
    predictions_a,
    predictions_b,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    predictions_a = np.asarray(
        predictions_a,
        dtype=np.int64,
    )

    predictions_b = np.asarray(
        predictions_b,
        dtype=np.int64,
    )

    correct_a = (
        predictions_a
        == labels
    )

    correct_b = (
        predictions_b
        == labels
    )

    a_correct_b_wrong = int(
        np.sum(
            correct_a
            & ~correct_b
        )
    )

    a_wrong_b_correct = int(
        np.sum(
            ~correct_a
            & correct_b
        )
    )

    discordant = (
        a_correct_b_wrong
        + a_wrong_b_correct
    )

    if discordant == 0:
        statistic = 0.0
        p_value = 1.0

    else:
        statistic = (
            (
                abs(
                    a_correct_b_wrong
                    - a_wrong_b_correct
                )
                - 1.0
            )
            ** 2
            / discordant
        )

        p_value = float(
            chi2.sf(
                statistic,
                df=1,
            )
        )


    return {
        "a_correct_b_wrong": (
            a_correct_b_wrong
        ),
        "a_wrong_b_correct": (
            a_wrong_b_correct
        ),
        "discordant_pairs": int(
            discordant
        ),
        "chi_square_statistic": float(
            statistic
        ),
        "raw_p_value": float(
            p_value
        ),
        "continuity_correction": True,
    }


def holm_adjust(
    raw_p_values,
):
    names = list(
        raw_p_values.keys()
    )

    values = np.asarray(
        [
            raw_p_values[
                name
            ]
            for name in names
        ],
        dtype=np.float64,
    )

    order = np.argsort(
        values
    )

    adjusted = np.empty_like(
        values
    )

    previous = 0.0

    family_size = len(
        names
    )


    for rank, index in enumerate(
        order
    ):
        candidate = (
            (
                family_size
                - rank
            )
            * values[
                index
            ]
        )

        candidate = min(
            1.0,
            max(
                previous,
                candidate,
            ),
        )

        adjusted[
            index
        ] = candidate

        previous = candidate


    return {
        names[index]: float(
            adjusted[
                index
            ]
        )
        for index
        in range(
            len(
                names
            )
        )
    }


def robustness_slice_metrics(
    labels,
    predictions,
    mask,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    predictions = np.asarray(
        predictions,
        dtype=np.int64,
    )

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    selected_labels = labels[
        mask
    ]

    selected_predictions = (
        predictions[
            mask
        ]
    )

    if selected_labels.size == 0:
        raise RuntimeError(
            "Robustness slice is empty."
        )

    (
        _,
        _,
        macro_f1,
        _,
    ) = (
        precision_recall_fscore_support(
            selected_labels,
            selected_predictions,
            average="macro",
            zero_division=0,
        )
    )

    accuracy = float(
        accuracy_score(
            selected_labels,
            selected_predictions,
        )
    )


    return {
        "count": int(
            selected_labels.size
        ),
        "fraction": float(
            selected_labels.size
            / labels.size
        ),
        "accuracy": accuracy,
        "error_rate": float(
            1.0
            - accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
    }


print(
    "BOOTSTRAP, McNEMAR, HOLM, AND SLICE FUNCTIONS DEFINED"
)


BOOTSTRAP, McNEMAR, HOLM, AND SLICE FUNCTIONS DEFINED


In [7]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

amp_enabled = bool(
    training_configuration[
        "training"
    ][
        "mixed_precision"
    ]
    and device.type == "cuda"
)

criterion = (
    nn.BCEWithLogitsLoss()
)


def construct_model_from_checkpoint(
    model_name,
):
    checkpoint_path = (
        CHECKPOINT_PATHS[
            model_name
        ]
    )

    actual_checkpoint_sha = (
        sha256_file(
            checkpoint_path
        )
    )

    if (
        actual_checkpoint_sha
        != EXPECTED_CHECKPOINT_SHA256[
            model_name
        ]
    ):
        raise RuntimeError(
            f"Checkpoint SHA mismatch: "
            f"{model_name}"
        )


    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True,
    )


    if (
        checkpoint[
            "model_name"
        ]
        != model_name
    ):
        raise RuntimeError(
            "Checkpoint model-name mismatch."
        )


    if (
        str(
            checkpoint[
                "config_sha256"
            ]
        ).lower()
        != EXPECTED_CONFIG_SHA256
    ):
        raise RuntimeError(
            "Checkpoint config SHA mismatch."
        )


    if (
        str(
            checkpoint[
                "vocabulary_sha256"
            ]
        ).lower()
        != EXPECTED_VOCABULARY_SHA256
    ):
        raise RuntimeError(
            "Checkpoint vocabulary SHA mismatch."
        )


    if (
        str(
            checkpoint[
                "split_manifest_sha256"
            ]
        ).lower()
        != EXPECTED_SPLIT_MANIFEST_SHA256
    ):
        raise RuntimeError(
            "Checkpoint split-manifest SHA mismatch."
        )


    if (
        float(
            checkpoint[
                "decision_threshold"
            ]
        )
        != 0.5
    ):
        raise RuntimeError(
            "Checkpoint decision threshold mismatch."
        )


    if (
        model_name
        == "baseline_mean_pool"
    ):
        model = (
            MeanPoolSentimentClassifier(
                vocabulary_size=int(
                    checkpoint[
                        "vocabulary_size"
                    ]
                ),
                embedding_dimension=int(
                    checkpoint[
                        "embedding_dimension"
                    ]
                ),
                padding_index=int(
                    checkpoint[
                        "pad_index"
                    ]
                ),
                dropout_probability=float(
                    checkpoint[
                        "dropout"
                    ]
                ),
            )
        )

    elif (
        model_name
        == "textcnn"
    ):
        model = (
            TextCNNSentimentClassifier(
                vocabulary_size=int(
                    checkpoint[
                        "vocabulary_size"
                    ]
                ),
                embedding_dimension=int(
                    checkpoint[
                        "embedding_dimension"
                    ]
                ),
                padding_index=int(
                    checkpoint[
                        "pad_index"
                    ]
                ),
                kernel_sizes=[
                    int(
                        value
                    )
                    for value
                    in checkpoint[
                        "kernel_sizes"
                    ]
                ],
                filters_per_kernel=int(
                    checkpoint[
                        "filters_per_kernel"
                    ]
                ),
                dropout_probability=float(
                    checkpoint[
                        "dropout"
                    ]
                ),
            )
        )

    elif (
        model_name
        == "bigru_attention"
    ):
        model = (
            BiGRUAttentionSentimentClassifier(
                vocabulary_size=int(
                    checkpoint[
                        "vocabulary_size"
                    ]
                ),
                embedding_dimension=int(
                    checkpoint[
                        "embedding_dimension"
                    ]
                ),
                padding_index=int(
                    checkpoint[
                        "pad_index"
                    ]
                ),
                hidden_dimension=int(
                    checkpoint[
                        "hidden_dimension"
                    ]
                ),
                recurrent_layers=int(
                    checkpoint[
                        "recurrent_layers"
                    ]
                ),
                bidirectional=bool(
                    checkpoint[
                        "bidirectional"
                    ]
                ),
                attention_dimension=int(
                    checkpoint[
                        "attention_dimension"
                    ]
                ),
                dropout_probability=float(
                    checkpoint[
                        "dropout"
                    ]
                ),
            )
        )

    else:
        raise KeyError(
            model_name
        )


    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True,
    )

    model = model.to(
        device
    )

    model.eval()


    parameter_count = sum(
        parameter.numel()
        for parameter
        in model.parameters()
    )


    if (
        parameter_count
        != EXPECTED_PARAMETER_COUNTS[
            model_name
        ]
    ):
        raise RuntimeError(
            f"Parameter-count mismatch: "
            f"{model_name}"
        )


    return (
        checkpoint,
        model,
    )


@torch.inference_mode()
def run_official_test_inference(
    model,
    input_ids,
    labels,
    batch_size,
):
    model.eval()

    total_examples = int(
        labels.shape[0]
    )

    total_loss = 0.0

    probability_chunks = []


    if device.type == "cuda":
        torch.cuda.synchronize()


    start_time = (
        time.perf_counter()
    )


    for start in range(
        0,
        total_examples,
        batch_size,
    ):
        end = min(
            total_examples,
            start + batch_size,
        )

        batch_inputs = (
            torch.from_numpy(
                np.array(
                    input_ids[
                        start:end
                    ],
                    dtype=np.int64,
                    copy=True,
                )
            )
            .to(
                device
            )
        )

        batch_labels = (
            torch.from_numpy(
                np.array(
                    labels[
                        start:end
                    ],
                    dtype=np.float32,
                    copy=True,
                )
            )
            .to(
                device
            )
        )


        with torch.autocast(
            device_type=(
                device.type
            ),
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            logits = model(
                batch_inputs
            )

            loss = criterion(
                logits,
                batch_labels,
            )


        # Preserve the training notebooks' inference convention:
        # sigmoid is applied after leaving the autocast context.
        probabilities = (
            torch.sigmoid(
                logits
            )
        )


        batch_count = int(
            batch_labels.shape[0]
        )

        total_loss += (
            float(
                loss.item()
            )
            * batch_count
        )

        probability_chunks.append(
            probabilities
            .detach()
            .cpu()
            .numpy()
        )


    if device.type == "cuda":
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start_time
    )


    probabilities = (
        np.concatenate(
            probability_chunks
        )
    )


    if probabilities.shape != (
        total_examples,
    ):
        raise RuntimeError(
            "Probability shape mismatch."
        )


    return {
        "probabilities": (
            probabilities
        ),
        "loss": float(
            total_loss
            / total_examples
        ),
        "evaluation_seconds": float(
            elapsed
        ),
        "examples_per_second": float(
            total_examples
            / elapsed
        ),
    }


print(
    "Device:",
    device,
)

print(
    "AMP enabled:",
    amp_enabled,
)

print(
    "MODEL RECONSTRUCTION AND INFERENCE FUNCTIONS DEFINED"
)


Device: cuda
AMP enabled: True
MODEL RECONSTRUCTION AND INFERENCE FUNCTIONS DEFINED


## Official-test access boundary

**The next code cell is the first deliberate official-test
array access in this project.**

Before that cell runs:

- all three checkpoints are fixed;
- decision threshold 0.5 is fixed;
- bootstrap settings are fixed;
- McNemar/Holm settings are fixed;
- robustness-slice thresholds are fixed;
- the final-evaluation protocol has already been committed
  and pushed.

Official-test results must not be used to retrain models or
revise any frozen threshold.


In [8]:
evaluation_started_utc = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


test_inputs_path = (
    PROCESSED
    / "test_input_ids.npy"
)

test_labels_path = (
    PROCESSED
    / "test_labels.npy"
)

test_source_indices_path = (
    PROCESSED
    / "test_source_indices.npy"
)


test_inputs = np.load(
    test_inputs_path,
    mmap_mode="r",
)

test_labels = np.load(
    test_labels_path,
    mmap_mode="r",
)

test_source_indices = np.load(
    test_source_indices_path,
    mmap_mode="r",
)


assert test_inputs.shape == (
    38_000,
    256,
)

assert test_labels.shape == (
    38_000,
)

assert test_source_indices.shape == (
    38_000,
)

assert test_inputs.dtype == np.int32


label_values = set(
    np.unique(
        test_labels
    ).tolist()
)

assert label_values == {
    0,
    1,
}


assert (
    np.unique(
        test_source_indices
    ).size
    == 38_000
)


pad_index = int(
    evaluation_protocol[
        "robustness_slices"
    ][
        "pad_index"
    ]
)

unknown_index = int(
    evaluation_protocol[
        "robustness_slices"
    ][
        "unknown_index"
    ]
)


test_input_array = np.asarray(
    test_inputs
)


effective_lengths = (
    test_input_array
    != pad_index
).sum(
    axis=1
).astype(
    np.int32
)


oov_counts = (
    test_input_array
    == unknown_index
).sum(
    axis=1
).astype(
    np.int32
)


if np.any(
    effective_lengths <= 0
):
    raise RuntimeError(
        "Official test contains zero-length encoded row."
    )


oov_rates = (
    oov_counts
    / effective_lengths
).astype(
    np.float64
)


short_threshold = int(
    evaluation_protocol[
        "robustness_slices"
    ][
        "short_reviews"
    ][
        "threshold"
    ]
)


medium_upper = int(
    evaluation_protocol[
        "robustness_slices"
    ][
        "medium_reviews"
    ][
        "upper_inclusive"
    ]
)


high_oov_threshold = float(
    evaluation_protocol[
        "robustness_slices"
    ][
        "high_oov_rate"
    ][
        "threshold"
    ]
)


negation_ids = np.asarray(
    list(
        evaluation_protocol[
            "robustness_slices"
        ][
            "contains_negation"
        ][
            "token_ids"
        ].values()
    ),
    dtype=np.int32,
)


slice_masks = {
    "short_reviews": (
        effective_lengths
        <= short_threshold
    ),
    "medium_reviews": (
        (
            effective_lengths
            > short_threshold
        )
        & (
            effective_lengths
            <= medium_upper
        )
    ),
    "long_reviews": (
        effective_lengths
        > medium_upper
    ),
    "contains_negation": (
        np.isin(
            test_input_array,
            negation_ids,
        ).any(
            axis=1
        )
    ),
    "high_oov_rate": (
        oov_rates
        >= high_oov_threshold
    ),
}


length_membership = (
    slice_masks[
        "short_reviews"
    ].astype(
        np.int8
    )
    + slice_masks[
        "medium_reviews"
    ].astype(
        np.int8
    )
    + slice_masks[
        "long_reviews"
    ].astype(
        np.int8
    )
)


if not np.all(
    length_membership == 1
):
    raise RuntimeError(
        "Length slices do not form exact partition."
    )


print(
    "OFFICIAL TEST ARRAYS LOADED"
)

print(
    "Examples:",
    len(
        test_labels
    ),
)

print(
    "Sequence length:",
    test_inputs.shape[
        1
    ],
)

print()

for (
    slice_name,
    mask,
) in slice_masks.items():
    print(
        slice_name,
        "count=",
        int(
            mask.sum()
        ),
        "fraction=",
        float(
            mask.mean()
        ),
    )


OFFICIAL TEST ARRAYS LOADED
Examples: 38000
Sequence length: 256

short_reviews count= 9996 fraction= 0.26305263157894737
medium_reviews count= 18591 fraction= 0.48923684210526314
long_reviews count= 9413 fraction= 0.24771052631578946
contains_negation count= 27951 fraction= 0.7355526315789473
high_oov_rate count= 2336 fraction= 0.06147368421052631


In [9]:
decision_threshold = float(
    evaluation_protocol[
        "classification"
    ][
        "decision_threshold"
    ]
)


test_probabilities = {}

test_predictions = {}

final_test_metrics = {}

inference_contract = {}


for model_name in MODEL_ORDER:
    print(
        "\nEvaluating:",
        model_name,
    )


    (
        checkpoint,
        model,
    ) = (
        construct_model_from_checkpoint(
            model_name
        )
    )


    batch_size = int(
        training_configuration[
            "models"
        ][
            model_name
        ][
            "batch_size"
        ]
    )


    inference_result = (
        run_official_test_inference(
            model=model,
            input_ids=test_inputs,
            labels=test_labels,
            batch_size=batch_size,
        )
    )


    probabilities = (
        inference_result[
            "probabilities"
        ]
    )


    predictions = (
        probabilities
        >= decision_threshold
    ).astype(
        np.int64
    )


    raw_metrics = (
        classification_metrics(
            np.asarray(
                test_labels
            ),
            probabilities,
            threshold=(
                decision_threshold
            ),
        )
    )


    metrics = canonical_metrics(
        raw_metrics
    )


    metrics[
        "loss"
    ] = float(
        inference_result[
            "loss"
        ]
    )

    metrics[
        "evaluation_seconds"
    ] = float(
        inference_result[
            "evaluation_seconds"
        ]
    )

    metrics[
        "examples_per_second"
    ] = float(
        inference_result[
            "examples_per_second"
        ]
    )

    metrics[
        "parameter_count"
    ] = int(
        EXPECTED_PARAMETER_COUNTS[
            model_name
        ]
    )


    test_probabilities[
        model_name
    ] = probabilities

    test_predictions[
        model_name
    ] = predictions

    final_test_metrics[
        model_name
    ] = metrics


    inference_contract[
        model_name
    ] = {
        "batch_size": (
            batch_size
        ),
        "amp_enabled": bool(
            amp_enabled
        ),
        "probability_dtype": str(
            probabilities.dtype
        ),
        "checkpoint_sha256": (
            EXPECTED_CHECKPOINT_SHA256[
                model_name
            ]
        ),
    }


    print(
        json.dumps(
            metrics,
            indent=2,
        )
    )


    del model
    del checkpoint


    if device.type == "cuda":
        torch.cuda.empty_cache()


comparison_table = pd.DataFrame(
    final_test_metrics
).T


display(
    comparison_table[
        [
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "mcc",
            "roc_auc",
            "pr_auc",
            "brier_score",
            "ece_15_bins",
            "loss",
            "examples_per_second",
            "parameter_count",
        ]
    ]
)



Evaluating: baseline_mean_pool


{
  "accuracy": 0.9294736842105263,
  "precision_macro": 0.9294736889692375,
  "precision_micro": 0.9294736842105263,
  "precision_weighted": 0.9294736889692378,
  "recall_macro": 0.9294736842105262,
  "recall_micro": 0.9294736842105263,
  "recall_weighted": 0.9294736842105263,
  "f1_macro": 0.9294736840151625,
  "f1_micro": 0.9294736842105263,
  "f1_weighted": 0.9294736840151626,
  "mcc": 0.8589473731797639,
  "roc_auc": 0.9776479916897507,
  "pr_auc": 0.9775760876007973,
  "brier_score": 0.05377662483133172,
  "ece_15_bins": 0.008339227256021996,
  "confusion_matrix": [
    [
      17661,
      1339
    ],
    [
      1341,
      17659
    ]
  ],
  "loss": 0.1925286834930119,
  "evaluation_seconds": 1.009150299942121,
  "examples_per_second": 37655.44141658528,
  "parameter_count": 6400129
}

Evaluating: textcnn


{
  "accuracy": 0.9417368421052632,
  "precision_macro": 0.9417544633359336,
  "precision_micro": 0.9417368421052632,
  "precision_weighted": 0.9417544633359336,
  "recall_macro": 0.9417368421052632,
  "recall_micro": 0.9417368421052632,
  "recall_weighted": 0.9417368421052632,
  "f1_macro": 0.9417362610818281,
  "f1_micro": 0.9417368421052632,
  "f1_weighted": 0.9417362610818281,
  "mcc": 0.883491305265469,
  "roc_auc": 0.9856706343490305,
  "pr_auc": 0.9860791532130821,
  "brier_score": 0.04481429661252991,
  "ece_15_bins": 0.017341390528176962,
  "confusion_matrix": [
    [
      17833,
      1167
    ],
    [
      1047,
      17953
    ]
  ],
  "loss": 0.1618210131808331,
  "evaluation_seconds": 2.430869299918413,
  "examples_per_second": 15632.267848080268,
  "parameter_count": 6597377
}

Evaluating: bigru_attention


{
  "accuracy": 0.9448684210526316,
  "precision_macro": 0.9450002311072356,
  "precision_micro": 0.9448684210526316,
  "precision_weighted": 0.9450002311072356,
  "recall_macro": 0.9448684210526316,
  "recall_micro": 0.9448684210526316,
  "recall_weighted": 0.9448684210526316,
  "f1_macro": 0.9448643382263312,
  "f1_micro": 0.9448684210526316,
  "f1_weighted": 0.9448643382263313,
  "mcc": 0.8898686423978138,
  "roc_auc": 0.9884153060941829,
  "pr_auc": 0.988652130829139,
  "brier_score": 0.04050608959962163,
  "ece_15_bins": 0.007061472972756902,
  "confusion_matrix": [
    [
      17789,
      1211
    ],
    [
      884,
      18116
    ]
  ],
  "loss": 0.1387595862903093,
  "evaluation_seconds": 5.930485300021246,
  "examples_per_second": 6407.57005162189,
  "parameter_count": 6631425
}


,accuracy,precision_macro,recall_macro,f1_macro,mcc,roc_auc,pr_auc,brier_score,ece_15_bins,loss,examples_per_second,parameter_count
baseline_mean_pool,0.929474,0.929474,0.929474,0.929474,0.858947,0.977648,0.977576,0.053777,0.008339,0.192529,37655.441417,6400129
textcnn,0.941737,0.941754,0.941737,0.941736,0.883491,0.985671,0.986079,0.044814,0.017341,0.161821,15632.267848,6597377
bigru_attention,0.944868,0.945,0.944868,0.944864,0.889869,0.988415,0.988652,0.040506,0.007061,0.13876,6407.570052,6631425


In [10]:
bootstrap_configuration = (
    evaluation_protocol[
        "bootstrap"
    ]
)


bootstrap_intervals = (
    bootstrap_confidence_intervals(
        labels=np.asarray(
            test_labels
        ),
        predictions_by_model=(
            test_predictions
        ),
        replicates=int(
            bootstrap_configuration[
                "replicates"
            ]
        ),
        confidence_level=float(
            bootstrap_configuration[
                "confidence_level"
            ]
        ),
        seed=int(
            bootstrap_configuration[
                "seed"
            ]
        ),
    )
)


for model_name in MODEL_ORDER:
    bootstrap_intervals[
        model_name
    ][
        "accuracy"
    ][
        "point_estimate"
    ] = float(
        final_test_metrics[
            model_name
        ][
            "accuracy"
        ]
    )

    bootstrap_intervals[
        model_name
    ][
        "macro_f1"
    ][
        "point_estimate"
    ] = float(
        final_test_metrics[
            model_name
        ][
            "f1_macro"
        ]
    )

    bootstrap_intervals[
        model_name
    ][
        "mcc"
    ][
        "point_estimate"
    ] = float(
        final_test_metrics[
            model_name
        ][
            "mcc"
        ]
    )


comparison_pairs = {
    "baseline_mean_pool_vs_textcnn": (
        "baseline_mean_pool",
        "textcnn",
    ),
    "baseline_mean_pool_vs_bigru_attention": (
        "baseline_mean_pool",
        "bigru_attention",
    ),
}


mcnemar_results = {}


for (
    comparison_name,
    (
        model_a,
        model_b,
    ),
) in comparison_pairs.items():
    result = (
        mcnemar_continuity_corrected(
            labels=np.asarray(
                test_labels
            ),
            predictions_a=(
                test_predictions[
                    model_a
                ]
            ),
            predictions_b=(
                test_predictions[
                    model_b
                ]
            ),
        )
    )

    result[
        "model_a"
    ] = model_a

    result[
        "model_b"
    ] = model_b

    mcnemar_results[
        comparison_name
    ] = result


adjusted_p_values = holm_adjust(
    {
        comparison_name: (
            result[
                "raw_p_value"
            ]
        )
        for (
            comparison_name,
            result,
        )
        in mcnemar_results.items()
    }
)


for comparison_name in (
    mcnemar_results
):
    adjusted = float(
        adjusted_p_values[
            comparison_name
        ]
    )

    mcnemar_results[
        comparison_name
    ][
        "holm_adjusted_p_value"
    ] = adjusted

    mcnemar_results[
        comparison_name
    ][
        "reject_after_holm_at_0_05"
    ] = bool(
        adjusted < 0.05
    )


slice_results = {}


for model_name in MODEL_ORDER:
    slice_results[
        model_name
    ] = {}

    for (
        slice_name,
        mask,
    ) in slice_masks.items():
        slice_results[
            model_name
        ][
            slice_name
        ] = (
            robustness_slice_metrics(
                labels=np.asarray(
                    test_labels
                ),
                predictions=(
                    test_predictions[
                        model_name
                    ]
                ),
                mask=mask,
            )
        )


print(
    "BOOTSTRAP CONFIDENCE INTERVALS"
)

print(
    json.dumps(
        bootstrap_intervals,
        indent=2,
    )
)


print(
    "\nPAIRED McNEMAR WITH HOLM CORRECTION"
)

print(
    json.dumps(
        mcnemar_results,
        indent=2,
    )
)


slice_rows = []


for model_name in MODEL_ORDER:
    for (
        slice_name,
        result,
    ) in slice_results[
        model_name
    ].items():
        slice_rows.append(
            {
                "model": model_name,
                "slice": slice_name,
                **result,
            }
        )


display(
    pd.DataFrame(
        slice_rows
    )
)


BOOTSTRAP CONFIDENCE INTERVALS
{
  "baseline_mean_pool": {
    "accuracy": {
      "lower": 0.9269730263157895,
      "upper": 0.9319736842105263,
      "point_estimate": 0.9294736842105263
    },
    "macro_f1": {
      "lower": 0.9269699196501038,
      "upper": 0.9319701264648411,
      "point_estimate": 0.9294736840151625
    },
    "mcc": {
      "lower": 0.8539446225551273,
      "upper": 0.8639412680345181,
      "point_estimate": 0.8589473731797639
    }
  },
  "textcnn": {
    "accuracy": {
      "lower": 0.9393151315789474,
      "upper": 0.9440526315789474,
      "point_estimate": 0.9417368421052632
    },
    "macro_f1": {
      "lower": 0.9393102254955834,
      "upper": 0.9440521896996616,
      "point_estimate": 0.9417362610818281
    },
    "mcc": {
      "lower": 0.8786393025643553,
      "upper": 0.8881258994357549,
      "point_estimate": 0.883491305265469
    }
  },
  "bigru_attention": {
    "accuracy": {
      "lower": 0.9425,
      "upper": 0.947078947368421,
   

,model,slice,count,fraction,accuracy,error_rate,macro_f1
0,baseline_mean_pool,short_reviews,9996,0.263053,0.924970,0.075030,0.922356
1,baseline_mean_pool,medium_reviews,18591,0.489237,0.931580,0.068420,0.931577
2,baseline_mean_pool,long_reviews,9413,0.247711,0.930097,0.069903,0.927679
3,baseline_mean_pool,contains_negation,27951,0.735553,0.924511,0.075489,0.921872
4,baseline_mean_pool,high_oov_rate,2336,0.061474,0.917808,0.082192,0.913829
5,textcnn,short_reviews,9996,0.263053,0.936475,0.063525,0.934229
6,textcnn,medium_reviews,18591,0.489237,0.945296,0.054704,0.945296
7,textcnn,long_reviews,9413,0.247711,0.940295,0.059705,0.938362
8,textcnn,contains_negation,27951,0.735553,0.940324,0.059676,0.938360
9,textcnn,high_oov_rate,2336,0.061474,0.928082,0.071918,0.924423


In [11]:
METRICS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

PLOTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTIONS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


def write_json_atomic(
    path,
    payload,
):
    temporary = Path(
        str(
            path
        )
        + ".tmp"
    )

    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary,
        path,
    )


git_head = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(
                ROOT
            ),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)


evaluation_completed_utc = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


final_metrics_payload = {
    "evaluation": {
        "official_test_used": True,
        "official_test_examples": int(
            len(
                test_labels
            )
        ),
        "evaluation_started_utc": (
            evaluation_started_utc
        ),
        "evaluation_completed_utc": (
            evaluation_completed_utc
        ),
        "git_head_at_evaluation": (
            git_head
        ),
        "protocol_sha256": (
            EXPECTED_PROTOCOL_SHA256
        ),
        "decision_threshold": (
            decision_threshold
        ),
        "device": str(
            device
        ),
        "amp_enabled": bool(
            amp_enabled
        ),
    },
    "models": (
        final_test_metrics
    ),
    "inference_contract": (
        inference_contract
    ),
    "training_resource_summary": (
        training_resource_summary
    ),
}


bootstrap_payload = {
    "method": (
        "nonparametric_percentile"
    ),
    "sampling": (
        "with_replacement"
    ),
    "replicate_sample_size": int(
        len(
            test_labels
        )
    ),
    "replicates": int(
        bootstrap_configuration[
            "replicates"
        ]
    ),
    "confidence_level": float(
        bootstrap_configuration[
            "confidence_level"
        ]
    ),
    "seed": int(
        bootstrap_configuration[
            "seed"
        ]
    ),
    "intervals": (
        bootstrap_intervals
    ),
}


mcnemar_payload = {
    "test": "mcnemar",
    "continuity_correction": True,
    "multiple_testing": {
        "method": "holm",
        "family_size": 2,
        "alpha": 0.05,
    },
    "comparisons": (
        mcnemar_results
    ),
}


robustness_payload = {
    "definitions": (
        evaluation_protocol[
            "robustness_slices"
        ]
    ),
    "models": (
        slice_results
    ),
}


final_metrics_path = (
    METRICS_DIRECTORY
    / "final_test_metrics.json"
)

bootstrap_path = (
    METRICS_DIRECTORY
    / "bootstrap_confidence_intervals.json"
)

mcnemar_path = (
    METRICS_DIRECTORY
    / "mcnemar_tests.json"
)

robustness_path = (
    METRICS_DIRECTORY
    / "robustness_slice_metrics.json"
)

predictions_path = (
    PREDICTIONS_DIRECTORY
    / "final_test_predictions.csv"
)


write_json_atomic(
    final_metrics_path,
    final_metrics_payload,
)

write_json_atomic(
    bootstrap_path,
    bootstrap_payload,
)

write_json_atomic(
    mcnemar_path,
    mcnemar_payload,
)

write_json_atomic(
    robustness_path,
    robustness_payload,
)


prediction_frame = pd.DataFrame(
    {
        "test_row": np.arange(
            len(
                test_labels
            ),
            dtype=np.int64,
        ),
        "source_index": np.asarray(
            test_source_indices,
            dtype=np.int64,
        ),
        "true_label": np.asarray(
            test_labels,
            dtype=np.int64,
        ),
        "effective_length": (
            effective_lengths
        ),
        "oov_count": (
            oov_counts
        ),
        "oov_rate": (
            oov_rates
        ),
    }
)


for (
    slice_name,
    mask,
) in slice_masks.items():
    prediction_frame[
        f"slice_{slice_name}"
    ] = mask.astype(
        np.int8
    )


for model_name in MODEL_ORDER:
    prediction_frame[
        f"{model_name}_probability"
    ] = np.asarray(
        test_probabilities[
            model_name
        ],
        dtype=np.float64,
    )

    prediction_frame[
        f"{model_name}_prediction"
    ] = (
        test_predictions[
            model_name
        ]
    )

    prediction_frame[
        f"{model_name}_correct"
    ] = (
        test_predictions[
            model_name
        ]
        == np.asarray(
            test_labels,
            dtype=np.int64,
        )
    ).astype(
        np.int8
    )


temporary_predictions = Path(
    str(
        predictions_path
    )
    + ".tmp"
)


prediction_frame.to_csv(
    temporary_predictions,
    index=False,
)

os.replace(
    temporary_predictions,
    predictions_path,
)


confusion_plot_paths = {}


for model_name in MODEL_ORDER:
    matrix = np.asarray(
        final_test_metrics[
            model_name
        ][
            "confusion_matrix"
        ],
        dtype=np.int64,
    )

    figure, axis = (
        plt.subplots(
            figsize=(
                5.5,
                4.5,
            )
        )
    )

    display_object = (
        ConfusionMatrixDisplay(
            confusion_matrix=matrix,
            display_labels=[
                "negative",
                "positive",
            ],
        )
    )

    display_object.plot(
        ax=axis,
        values_format="d",
        colorbar=False,
    )

    axis.set_title(
        (
            f"{model_name} ? "
            "Official Test Confusion Matrix"
        )
    )

    figure.tight_layout()

    path = (
        PLOTS_DIRECTORY
        / (
            f"{model_name}_"
            "final_test_confusion_matrix.png"
        )
    )

    figure.savefig(
        path,
        dpi=160,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )

    confusion_plot_paths[
        model_name
    ] = str(
        path.relative_to(
            ROOT
        )
    )


print(
    "FINAL TEST ARTIFACTS WRITTEN"
)

for path in [
    final_metrics_path,
    bootstrap_path,
    mcnemar_path,
    robustness_path,
    predictions_path,
]:
    print(
        path.relative_to(
            ROOT
        ),
        "| bytes:",
        path.stat().st_size,
    )


for (
    model_name,
    path,
) in confusion_plot_paths.items():
    print(
        model_name,
        "=>",
        path,
    )


print()
print(
    "FINAL OFFICIAL-TEST EVALUATION COMPLETE"
)


FINAL TEST ARTIFACTS WRITTEN
task2_sentiment\Anshika_Goel\outputs\metrics\final_test_metrics.json | bytes: 4944
task2_sentiment\Anshika_Goel\outputs\metrics\bootstrap_confidence_intervals.json | bytes: 1653
task2_sentiment\Anshika_Goel\outputs\metrics\mcnemar_tests.json | bytes: 1092
task2_sentiment\Anshika_Goel\outputs\metrics\robustness_slice_metrics.json | bytes: 5524
task2_sentiment\Anshika_Goel\outputs\predictions\final_test_predictions.csv | bytes: 3590911
baseline_mean_pool => task2_sentiment\Anshika_Goel\outputs\plots\baseline_mean_pool_final_test_confusion_matrix.png
textcnn => task2_sentiment\Anshika_Goel\outputs\plots\textcnn_final_test_confusion_matrix.png
bigru_attention => task2_sentiment\Anshika_Goel\outputs\plots\bigru_attention_final_test_confusion_matrix.png

FINAL OFFICIAL-TEST EVALUATION COMPLETE


## Interpretation boundary

The results above constitute the common final test evaluation.
They must not be used to retrain a model or revise any frozen
threshold.

Notebook 06 will use the saved per-example prediction table to
perform the required 20-example manual error review:

- 5 confident false positives;
- 5 confident false negatives;
- 5 near-threshold errors;
- 5 slice-specific failures.
